# Plotting posterior biases in PE-uncertainty units

Transforming every posterior coordinate as

\[
\frac{\theta-\theta_{\mathrm{true}}}{\sigma_{\mathrm{PE}}},
\]

then producing multidimensional corner plots and tables of the mean and median normalized biases for every active FTI case.


## Importing the required packages and setting the directories


In [1]:
import json
import re
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import lisabeta.utils.plotutils as plotutils

try:
    import corner
except ImportError as error:
    raise ImportError(
        "The Python package 'corner' is required. "
        "Install it in the lisabeta environment with: pip install corner"
    ) from error

from IPython.display import display

# Keeping Matplotlib independent of an external LaTeX installation even when imported settings enable it globally.
plt.rcParams['text.usetex'] = False
plt.rcParams['mathtext.fontset'] = 'dejavusans'


/pbs/home/a/amahdi/lisabeta/src/lisabeta/waveforms/bbh/lalsim_wrap.py:10: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal


In [2]:
# Defining run paths, output directories, and normalization controls.

PE_BASE_DIR = Path('/pbs/home/a/amahdi/PE_for_my_work')
CASE_ID = 'M1e6_SEOBv5_id_6' # Change the id number according to your deviation value

CONFIG_DIR = PE_BASE_DIR / 'config_files' / CASE_ID

# Saving the normalized plots inside the selected configuration directory without reusing any existing corner-plot filename.
NORMALIZED_CORNER_ROOT = (
    CONFIG_DIR
    / 'normalized_bias_corner_plots'
)
FULL_CORNER_DIR = (
    NORMALIZED_CORNER_ROOT
    / 'full_corner_plots'
)
TABLE_DIR = NORMALIZED_CORNER_ROOT / 'tables'

for directory in [
    NORMALIZED_CORNER_ROOT,
    FULL_CORNER_DIR,
    TABLE_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

SIGMA_METHOD = 'std'
# Using the posterior sample standard deviation for `std`, or half of the 16th-to-84th percentile interval for `central68`.

BAD_LNL_CUT = -1e12
MAX_CORNER_SAMPLES = 30000
CORNER_BINS = 25
RANDOM_SEED = 20260713

config_files = sorted(CONFIG_DIR.glob('*.json'))

print('Config directory:')
print(CONFIG_DIR)
print()
print('Normalized corner plots will be saved only inside:')
print(NORMALIZED_CORNER_ROOT)
print()
print('Number of JSON files:', len(config_files))

if not config_files:
    raise FileNotFoundError(
        f'No JSON files were found in {CONFIG_DIR}. '
        'Check PE_BASE_DIR and CASE_ID.'
    )


Config directory:
/pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_6

Normalized corner plots will be saved only inside:
/pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_6/normalized_bias_corner_plots

Number of JSON files: 12


## Locating and loading the parameter-estimation outputs

Searching both the lisabeta string-concatenated output names and the ordinary joined-path alternatives, then loading the processed and raw posterior files available for each JSON configuration.


In [3]:
def infer_pe_base_dir_from_config_location(config_file):
    config_file = Path(config_file)
    return config_file.parent.parent.parent


def infer_run_data_root_from_config_location(config_file):
    return (
        infer_pe_base_dir_from_config_location(config_file)
        / 'run_data'
    )


def infer_run_dir_from_config_location(config_file):
    config_file = Path(config_file)
    return (
        infer_run_data_root_from_config_location(config_file)
        / config_file.parent.name
    )


def unique_labeled_paths(paths):
    unique = []
    seen = set()

    for label, path in paths:
        path = Path(path)
        key = str(path)

        if key not in seen:
            unique.append((label, path))
            seen.add(key)

    return unique


def candidate_output_paths(
    config_file,
    out_dir,
    out_name,
    suffix,
):
    config_file = Path(config_file)
    out_dir = Path(out_dir)

    run_data_root = (
        infer_run_data_root_from_config_location(config_file)
    )
    inferred_run_dir = (
        infer_run_dir_from_config_location(config_file)
    )
    case_id = config_file.parent.name

    candidates = [
        (
            'configured_string_concat',
            Path(str(out_dir) + f'{out_name}{suffix}'),
        ),
        (
            'inferred_string_concat',
            run_data_root / f'{case_id}{out_name}{suffix}',
        ),
        (
            'configured_path_join',
            out_dir / f'{out_name}{suffix}',
        ),
        (
            'inferred_path_join',
            inferred_run_dir / f'{out_name}{suffix}',
        ),
    ]

    return unique_labeled_paths(candidates)


def first_existing_path(candidates):
    for label, path in candidates:
        if path.exists():
            return label, path

    return None, None


def detect_active_fti_parameter(config):
    infer_params = list(
        config.get('prior_params', {})
        .get('infer_params', [])
    )

    configured_fti = (
        config.get('waveform_params', {})
        .get('tgr_params', {})
        .get('FTI', {})
        .get('fti_params', [])
    )

    active = [
        parameter
        for parameter in infer_params
        if parameter in configured_fti
        or parameter.startswith('fti_')
    ]

    if active:
        return active[-1]

    return None


def read_case_information(config_file):
    config_file = Path(config_file)

    with config_file.open('r') as stream:
        config = json.load(stream)

    run_params = config.get('run_params', {})
    infer_params = list(
        config.get('prior_params', {})
        .get('infer_params', [])
    )

    out_dir = Path(run_params.get('out_dir', ''))
    out_name = run_params.get(
        'out_name',
        config_file.stem,
    )

    processed_candidates = candidate_output_paths(
        config_file,
        out_dir,
        out_name,
        '.h5',
    )
    raw_candidates = candidate_output_paths(
        config_file,
        out_dir,
        out_name,
        '_raw.h5',
    )

    processed_kind, processed_file = (
        first_existing_path(processed_candidates)
    )
    raw_kind, raw_file = (
        first_existing_path(raw_candidates)
    )

    return {
        'config_file': config_file,
        'config': config,
        'run_params': run_params,
        'infer_params': infer_params,
        'active_fti': detect_active_fti_parameter(config),
        'out_dir': out_dir,
        'out_name': out_name,
        'processed_candidates': processed_candidates,
        'raw_candidates': raw_candidates,
        'processed_kind': processed_kind,
        'processed_file': processed_file,
        'raw_kind': raw_kind,
        'raw_file': raw_file,
    }


def load_raw_file(raw_file, infer_params):
    posterior = {}

    with h5py.File(raw_file, 'r') as h5:
        for parameter in infer_params:
            if parameter in h5:
                posterior[parameter] = h5[parameter][:]
            else:
                print(
                    f'[warning] {parameter} is absent '
                    f'from {raw_file}'
                )

        if 'lnlike' in h5:
            posterior['lnL'] = h5['lnlike'][:]
        elif 'lnL' in h5:
            posterior['lnL'] = h5['lnL'][:]
        else:
            raise KeyError(
                f"Neither 'lnlike' nor 'lnL' exists in {raw_file}"
            )

    return posterior


def load_case(config_file):
    case = read_case_information(config_file)

    case['processed_posterior'] = None
    case['raw_posterior'] = None

    if case['processed_file'] is not None:
        try:
            case['processed_posterior'] = (
                plotutils.load_params_posterior_lisa_smbh(
                    str(case['config_file']),
                    str(case['processed_file']),
                )
            )
        except Exception as error:
            print(
                '[warning] processed posterior could not be loaded '
                f"for {case['active_fti']}: {error}"
            )

    if case['raw_file'] is not None:
        try:
            case['raw_posterior'] = load_raw_file(
                case['raw_file'],
                case['infer_params'],
            )
        except Exception as error:
            print(
                '[warning] raw posterior could not be loaded '
                f"for {case['active_fti']}: {error}"
            )

    return case


In [4]:
cases = [load_case(path) for path in config_files]

for case in cases:
    print('=' * 100)
    print('Active FTI parameter:', case['active_fti'])
    print('Config:', case['config_file'])
    print('Processed posterior:', case['processed_file'])
    print('Raw posterior:', case['raw_file'])
    print('Inferred parameters:', case['infer_params'])

    if (
        case['processed_posterior'] is None
        and case['raw_posterior'] is None
    ):
        print('[warning] no usable posterior was loaded')


Active FTI parameter: fti_dchi0v
Config: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_6/M1e6_SEOBv5_id_6_fti_dchi0v_IMR.json
Processed posterior: /pbs/home/a/amahdi/PE_for_my_work/run_data/M1e6_SEOBv5_id_6M1e6_SEOBv5_id_6_fti_dchi0v_IMR.h5
Raw posterior: /pbs/home/a/amahdi/PE_for_my_work/run_data/M1e6_SEOBv5_id_6M1e6_SEOBv5_id_6_fti_dchi0v_IMR_raw.h5
Inferred parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'Deltat', 'dist', 'inc', 'phi', 'lambda', 'beta', 'psi', 'fti_dchi0v']
Active FTI parameter: fti_dchi1v
Config: /pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_6/M1e6_SEOBv5_id_6_fti_dchi1v_IMR.json
Processed posterior: /pbs/home/a/amahdi/PE_for_my_work/run_data/M1e6_SEOBv5_id_6M1e6_SEOBv5_id_6_fti_dchi1v_IMR.h5
Raw posterior: /pbs/home/a/amahdi/PE_for_my_work/run_data/M1e6_SEOBv5_id_6M1e6_SEOBv5_id_6_fti_dchi1v_IMR_raw.h5
Inferred parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'Deltat', 'dist', 'inc', 'phi', 'lambda', 'beta', 'psi', 'fti_dchi1v']
Active FTI

## Extracting the posterior samples

Using the processed posterior only when it contains every requested coordinate, including the active FTI parameter. Falling back to the raw chain when necessary so the active FTI parameter is never removed silently.


In [5]:
def processed_has_parameter(case, parameter):
    processed = case.get('processed_posterior')

    if processed is None:
        return False

    try:
        return parameter in processed['post']
    except Exception:
        return False


def raw_has_parameter(case, parameter):
    raw = case.get('raw_posterior')

    try:
        return raw is not None and parameter in raw
    except Exception:
        return False


def choose_corner_source(case, parameters):
    active_fti = case['active_fti']

    processed_available = [
        parameter
        for parameter in parameters
        if processed_has_parameter(case, parameter)
    ]
    raw_available = [
        parameter
        for parameter in parameters
        if raw_has_parameter(case, parameter)
    ]

    if (
        active_fti in processed_available
        and len(processed_available) == len(parameters)
    ):
        return 'processed', processed_available

    if active_fti in raw_available:
        missing = [
            parameter
            for parameter in parameters
            if parameter not in raw_available
        ]

        if missing:
            print(
                f'[warning] raw posterior for {active_fti} '
                f'is missing {missing}'
            )

        return 'raw', raw_available

    if active_fti in processed_available:
        missing = [
            parameter
            for parameter in parameters
            if parameter not in processed_available
        ]

        if missing:
            print(
                f'[warning] processed posterior for {active_fti} '
                f'is missing {missing}'
            )

        return 'processed', processed_available

    print(
        f'[error] active FTI parameter {active_fti} is absent '
        'from both available posterior sources'
    )
    return None, []


def decide_raw_burnin(case):
    raw = case.get('raw_posterior')

    if raw is None or 'lnL' not in raw:
        return 0, 'no raw likelihood'

    number_of_steps = int(
        np.asarray(raw['lnL']).shape[-1]
    )

    run_params = case.get('run_params', {})

    try:
        n_iter = (
            int(run_params['n_iter'])
            if run_params.get('n_iter') is not None
            else None
        )
    except Exception:
        n_iter = None

    try:
        burn_in = int(
            run_params.get('burn_in', 0) or 0
        )
    except Exception:
        burn_in = 0

    if burn_in <= 0:
        return 0, 'burn_in is zero or missing'

    if burn_in >= number_of_steps:
        return 0, (
            f'burn_in={burn_in} is not smaller than '
            f'raw nsteps={number_of_steps}'
        )

    if (
        n_iter is not None
        and number_of_steps == n_iter
    ):
        return burn_in, (
            f'raw nsteps={number_of_steps} equals '
            f'configured n_iter={n_iter}'
        )

    if (
        n_iter is not None
        and number_of_steps == max(n_iter - burn_in, 0)
    ):
        return 0, (
            'raw posterior already appears to have burn-in '
            'removed'
        )

    return 0, (
        'raw-chain length is ambiguous; no additional '
        'burn-in was removed'
    )


def slice_last_axis(array, start=0):
    array = np.asarray(array)

    index = [slice(None)] * array.ndim
    index[-1] = slice(start, None)

    return array[tuple(index)]


def align_array_to_likelihood(array, likelihood):
    array = np.asarray(array)
    likelihood = np.asarray(likelihood)

    if array.shape == likelihood.shape:
        return array

    if array.size == likelihood.size:
        return array.reshape(likelihood.shape)

    return None


def extract_processed_matrix(case, parameters):
    post = case['processed_posterior']['post']

    arrays = [
        np.asarray(post[parameter]).reshape(-1)
        for parameter in parameters
    ]

    lengths = [array.size for array in arrays]

    if len(set(lengths)) != 1:
        raise ValueError(
            'Processed posterior parameter lengths differ: '
            f'{dict(zip(parameters, lengths))}'
        )

    finite_mask = np.ones(lengths[0], dtype=bool)

    for array in arrays:
        finite_mask &= np.isfinite(array)

    samples = np.column_stack(
        [array[finite_mask] for array in arrays]
    )

    return samples, {
        'source': 'processed',
        'burnin_used': 0,
        'burnin_reason': 'processed posterior',
    }


def extract_raw_matrix(case, parameters):
    raw = case['raw_posterior']
    likelihood_full = np.asarray(raw['lnL'])

    burnin, burnin_reason = decide_raw_burnin(case)

    likelihood = slice_last_axis(
        likelihood_full,
        start=burnin,
    )

    mask = (
        np.isfinite(likelihood)
        & (likelihood > BAD_LNL_CUT)
    )

    arrays = []

    for parameter in parameters:
        aligned = align_array_to_likelihood(
            raw[parameter],
            likelihood_full,
        )

        if aligned is None:
            raise ValueError(
                f"{parameter} with shape "
                f"{np.asarray(raw[parameter]).shape} cannot "
                f"be aligned to likelihood shape "
                f"{likelihood_full.shape}"
            )

        sliced = slice_last_axis(
            aligned,
            start=burnin,
        )

        mask &= np.isfinite(sliced)
        arrays.append(sliced)

    samples = np.column_stack(
        [array[mask] for array in arrays]
    )

    return samples, {
        'source': 'raw',
        'burnin_used': burnin,
        'burnin_reason': burnin_reason,
    }


def extract_corner_matrix(case, source, parameters):
    if source == 'processed':
        return extract_processed_matrix(
            case,
            parameters,
        )

    if source == 'raw':
        return extract_raw_matrix(
            case,
            parameters,
        )

    raise ValueError(f'Unknown source: {source}')


## Calculating the injected values and posterior uncertainties

Constructing the normalized sample matrix one coordinate at a time by subtracting the corresponding injected value and dividing by its posterior uncertainty. Recording the posterior mean, median, central percentiles, and normalized biases in the accompanying table.


In [6]:
def get_true_value(case, parameter):
    processed = case.get('processed_posterior')

    if processed is not None:
        injected = processed.get('injparams_Lframe', {})

        try:
            if parameter in injected:
                value = float(
                    np.asarray(injected[parameter]).squeeze()
                )

                if np.isfinite(value):
                    return value
        except Exception:
            pass

    config = case['config']
    source_params = config.get('source_params', {})

    try:
        if parameter in source_params:
            value = float(
                np.asarray(source_params[parameter]).squeeze()
            )

            if np.isfinite(value):
                return value
    except Exception:
        pass

    fti_block = (
        config.get('waveform_params', {})
        .get('tgr_params', {})
        .get('FTI', {})
    )

    if (
        parameter == 'Mchirp'
        and fti_block.get('Mchirp_inj') is not None
    ):
        return float(fti_block['Mchirp_inj'])

    if (
        parameter == 'fti_dxi0v'
        and fti_block.get('dxi0v_inj') is not None
    ):
        return float(fti_block['dxi0v_inj'])

    if parameter == case.get('active_fti'):
        return 0.0

    return None


def calculate_sigma(samples):
    samples = np.asarray(samples)

    if samples.size < 2:
        return np.nan

    if SIGMA_METHOD == 'std':
        return float(
            np.std(samples, ddof=1)
        )

    if SIGMA_METHOD == 'central68':
        percentile_16, percentile_84 = np.percentile(
            samples,
            [16.0, 84.0],
        )
        return float(
            0.5
            * (percentile_84 - percentile_16)
        )

    raise ValueError(
        "SIGMA_METHOD must be 'std' or 'central68'"
    )


def make_normalized_matrix(
    case,
    samples,
    parameters,
):
    normalized_columns = []
    retained_parameters = []
    metadata_rows = []

    for column_index, parameter in enumerate(parameters):
        parameter_samples = samples[:, column_index]

        true_value = get_true_value(
            case,
            parameter,
        )

        if (
            true_value is None
            or not np.isfinite(true_value)
        ):
            print(
                f'[warning] skipping {parameter}: '
                'true value was not found'
            )
            continue

        sigma_pe = calculate_sigma(
            parameter_samples
        )

        if (
            not np.isfinite(sigma_pe)
            or sigma_pe <= 0
        ):
            print(
                f'[warning] skipping {parameter}: '
                f'invalid sigma_PE={sigma_pe}'
            )
            continue

        normalized = (
            parameter_samples - true_value
        ) / sigma_pe

        normalized_columns.append(normalized)
        retained_parameters.append(parameter)

        percentile_16, median, percentile_84 = np.percentile(
            parameter_samples,
            [16.0, 50.0, 84.0],
        )

        metadata_rows.append({
            'active_fti_run': case['active_fti'],
            'parameter': parameter,
            'true_value': true_value,
            'sigma_method': SIGMA_METHOD,
            'sigma_PE': sigma_pe,
            'posterior_mean': float(
                np.mean(parameter_samples)
            ),
            'posterior_median': float(median),
            'median_minus_true': float(
                median - true_value
            ),
            'posterior_percentile_16': float(
                percentile_16
            ),
            'posterior_percentile_84': float(
                percentile_84
            ),
            'mean_bias_over_sigma_PE': float(
                (
                    np.mean(parameter_samples)
                    - true_value
                )
                / sigma_pe
            ),
            'median_bias_over_sigma_PE': float(
                (median - true_value)
                / sigma_pe
            ),
        })

    if len(normalized_columns) < 2:
        raise ValueError(
            'Fewer than two parameters have both a true value '
            'and a valid posterior sigma.'
        )

    normalized_matrix = np.column_stack(
        normalized_columns
    )

    return (
        normalized_matrix,
        retained_parameters,
        metadata_rows,
    )


def normalized_axis_label(parameter):
    # Building the axis labels with Matplotlib mathtext while keeping the source text ASCII-safe.
    safe_parameter = str(parameter).replace('_', r'\_')
    return (
        rf'$\Delta\,\mathrm{{{safe_parameter}}}'
        rf'/\sigma_{{\mathrm{{PE}}}}$'
    )


def downsample_rows(matrix):
    if (
        MAX_CORNER_SAMPLES is None
        or len(matrix) <= MAX_CORNER_SAMPLES
    ):
        return matrix

    random_generator = np.random.default_rng(
        RANDOM_SEED
    )

    indices = random_generator.choice(
        len(matrix),
        size=MAX_CORNER_SAMPLES,
        replace=False,
    )

    return matrix[indices]


## Producing the normalized multidimensional corner plots

Placing the injected value at zero on every transformed axis, plotting the normalized posterior density, and saving both PNG and vector PDF versions in a separate output directory.


In [7]:
summary_rows = []
plot_rows = []

for case in cases:
    active_fti = case['active_fti']
    requested_parameters = list(
        dict.fromkeys(case['infer_params'])
    )

    if (
        active_fti
        and active_fti not in requested_parameters
    ):
        requested_parameters.append(active_fti)

    source, available_parameters = choose_corner_source(
        case,
        requested_parameters,
    )

    print()
    print('#' * 100)
    print('Active FTI run:', active_fti)
    print('Requested parameters:', requested_parameters)
    print('Selected source:', source)
    print('Available parameters:', available_parameters)

    if source is None:
        print(
            '[warning] no normalized corner plot was created '
            f'for {active_fti}'
        )
        continue

    samples, source_metadata = extract_corner_matrix(
        case,
        source,
        available_parameters,
    )

    (
        normalized_matrix,
        retained_parameters,
        parameter_rows,
    ) = make_normalized_matrix(
        case,
        samples,
        available_parameters,
    )

    normalized_for_plot = downsample_rows(
        normalized_matrix
    )

    labels = [
        normalized_axis_label(parameter)
        for parameter in retained_parameters
    ]

    truths = np.zeros(
        len(retained_parameters)
    )

    # Reapplying the local Matplotlib setting in case an imported library changed it.
    plt.rcParams['text.usetex'] = False

    figure = corner.corner(
        normalized_for_plot,

        labels=labels,
        truths=truths,
        truth_color='black',

        # Using one fixed colour consistently across the histograms and contours.
        color='tab:blue',

        # Using a finer histogram grid and Gaussian smoothing for the posterior density.
        bins=60,
        smooth=1.0,
        smooth1d=1.0,

        show_titles=True,
        title_fmt='.2f',
        quantiles=[0.16, 0.50, 0.84],

        # Displaying the smoothed density and contours without plotting individual samples.
        plot_datapoints=False,
        plot_density=True,
        plot_contours=True,
        fill_contours=True,

        max_n_ticks=3,

        label_kwargs={
            'fontsize': 8,
        },

        title_kwargs={
            'fontsize': 8,
        },

        hist_kwargs={
            'color': 'tab:blue',
            'linewidth': 1.4,
        },

        contour_kwargs={
            'colors': ['tab:blue'],
            'linewidths': 1.2,
        },

        contourf_kwargs={
            'alpha': 0.55,
        },

        levels=(
            1.0 - np.exp(-0.5),
            1.0 - np.exp(-2.0),
        ),
    )


    figure.suptitle(
        (
            f'{active_fti}: normalized posterior corner\n'
            f'({source} posterior; '
            f'sigma method = {SIGMA_METHOD})'
        ),
        y=1.01,
        usetex=False,
    )

    output_file = (
        FULL_CORNER_DIR
        / (
            f'{active_fti}'
            f'_all_parameters_normalized_bias_corner'
            f'_{source}.png'
        )
    )

    figure.savefig(
        output_file,
        dpi=350,
        bbox_inches='tight',
        pad_inches=0.12,
    )

    # Saving a vector PDF in addition to the PNG for lossless zooming and cropping.
    pdf_output_file = output_file.with_suffix('.pdf')
    figure.savefig(
        pdf_output_file,
        bbox_inches='tight',
        pad_inches=0.12,
    )

    print('Saved normalized corner plot:')
    print(output_file)

    display(figure)
    plt.close(figure)

    for row in parameter_rows:
        row['posterior_source'] = source
        row['raw_burnin_used'] = (
            source_metadata['burnin_used']
        )
        row['raw_burnin_reason'] = (
            source_metadata['burnin_reason']
        )
        summary_rows.append(row)

    plot_rows.append({
        'active_fti_run': active_fti,
        'posterior_source': source,
        'number_of_parameters': len(
            retained_parameters
        ),
        'parameters': ','.join(
            retained_parameters
        ),
        'number_of_samples_for_statistics': int(
            len(normalized_matrix)
        ),
        'number_of_samples_plotted': int(
            len(normalized_for_plot)
        ),
        'plot_file': str(output_file),
        'pdf_plot_file': str(pdf_output_file),
    })

summary_table = pd.DataFrame(summary_rows)
summary_file = (
    TABLE_DIR
    / 'normalized_corner_parameter_summary.csv'
)
summary_table.to_csv(
    summary_file,
    index=False,
)

plot_table = pd.DataFrame(plot_rows)
plot_table_file = (
    TABLE_DIR
    / 'normalized_corner_plot_files.csv'
)
plot_table.to_csv(
    plot_table_file,
    index=False,
)

# Creating a dedicated table for the median-based normalized bias.
median_bias_columns = [
    'active_fti_run',
    'parameter',
    'posterior_source',
    'true_value',
    'posterior_median',
    'median_minus_true',
    'sigma_method',
    'sigma_PE',
    'median_bias_over_sigma_PE',
]

median_bias_table = summary_table[
    median_bias_columns
].copy()

median_bias_file = (
    TABLE_DIR
    / 'median_bias_over_sigma_PE.csv'
)

median_bias_table.to_csv(
    median_bias_file,
    index=False,
)

print()
print('=' * 100)
print('Saved parameter summary:')
print(summary_file)
print()
print('Saved median-based normalized bias table:')
print(median_bias_file)
print()
print('Saved plot list:')
print(plot_table_file)

display(median_bias_table)
display(summary_table)
display(plot_table)



####################################################################################################
Active FTI run: fti_dchi0v
Requested parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'Deltat', 'dist', 'inc', 'phi', 'lambda', 'beta', 'psi', 'fti_dchi0v']
Selected source: processed
Available parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'Deltat', 'dist', 'inc', 'phi', 'lambda', 'beta', 'psi', 'fti_dchi0v']


findfont: Font family 'Times New Roman' not found.
findfont: Font family ['Times New Roman'] not found. Falling back to DejaVu Sans.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.
findfont: Font family 'Times New Roman' not found.


Saved normalized corner plot:
/pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_6/normalized_bias_corner_plots/full_corner_plots/fti_dchi0v_all_parameters_normalized_bias_corner_processed.png


<Figure size 2650x2650 with 144 Axes>


####################################################################################################
Active FTI run: fti_dchi1v
Requested parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'Deltat', 'dist', 'inc', 'phi', 'lambda', 'beta', 'psi', 'fti_dchi1v']
Selected source: processed
Available parameters: ['Mchirp', 'q', 'chiPN', 'chim', 'Deltat', 'dist', 'inc', 'phi', 'lambda', 'beta', 'psi', 'fti_dchi1v']


...

<Figure size 2650x2650 with 144 Axes>


Saved parameter summary:
/pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_6/normalized_bias_corner_plots/tables/normalized_corner_parameter_summary.csv

Saved median-based normalized bias table:
/pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_6/normalized_bias_corner_plots/tables/median_bias_over_sigma_PE.csv

Saved plot list:
/pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_6/normalized_bias_corner_plots/tables/normalized_corner_plot_files.csv


,active_fti_run,parameter,posterior_source,true_value,posterior_median,median_minus_true,sigma_method,sigma_PE,median_bias_over_sigma_PE
0,fti_dchi0v,Mchirp,processed,333021.282961,332882.988700,-138.294261,std,91.272709,-1.515176
1,fti_dchi0v,q,processed,4.000000,3.864336,-0.135664,std,0.004409,-30.767008
2,fti_dchi0v,chiPN,processed,0.446195,0.439037,-0.007158,std,0.000303,-23.584548
3,fti_dchi0v,chim,processed,0.300000,0.281272,-0.018728,std,0.002546,-7.357053
4,fti_dchi0v,Deltat,processed,0.000000,1.295835,1.295835,std,0.100309,12.918438
...,...,...,...,...,...,...,...,...,...
127,fti_dxi0v,phi,processed,0.700000,0.269226,-0.430774,std,0.015261,-28.226367
128,fti_dxi0v,lambda,processed,1.000000,0.970315,-0.029685,std,0.003566,-8.323545
129,fti_dxi0v,beta,processed,0.523599,0.518480,-0.005119,std,0.001863,-2.746903
130,fti_dxi0v,psi,processed,1.200000,1.165908,-0.034092,std,0.003969,-8.589205


,active_fti_run,parameter,true_value,sigma_method,sigma_PE,posterior_mean,posterior_median,median_minus_true,posterior_percentile_16,posterior_percentile_84,mean_bias_over_sigma_PE,median_bias_over_sigma_PE,posterior_source,raw_burnin_used,raw_burnin_reason
0,fti_dchi0v,Mchirp,333021.282961,std,91.272709,332883.406657,332882.988700,-138.294261,332792.856985,332974.039436,-1.510597,-1.515176,processed,0,processed posterior
1,fti_dchi0v,q,4.000000,std,0.004409,3.864335,3.864336,-0.135664,3.859962,3.868704,-30.767357,-30.767008,processed,0,processed posterior
2,fti_dchi0v,chiPN,0.446195,std,0.000303,0.439032,0.439037,-0.007158,0.438727,0.439337,-23.601407,-23.584548,processed,0,processed posterior
3,fti_dchi0v,chim,0.300000,std,0.002546,0.281263,0.281272,-0.018728,0.278710,0.283793,-7.360949,-7.357053,processed,0,processed posterior
4,fti_dchi0v,Deltat,0.000000,std,0.100309,1.296212,1.295835,1.295835,1.197327,1.396059,12.922194,12.918438,processed,0,processed posterior
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,fti_dxi0v,phi,0.700000,std,0.015261,0.269213,0.269226,-0.430774,0.254025,0.284412,-28.227228,-28.226367,processed,0,processed posterior
128,fti_dxi0v,lambda,1.000000,std,0.003566,0.970290,0.970315,-0.029685,0.966788,0.973822,-8.330607,-8.323545,processed,0,processed posterior
129,fti_dxi0v,beta,0.523599,std,0.001863,0.518465,0.518480,-0.005119,0.516607,0.520297,-2.755159,-2.746903,processed,0,processed posterior
130,fti_dxi0v,psi,1.200000,std,0.003969,1.165911,1.165908,-0.034092,1.162013,1.169885,-8.588493,-8.589205,processed,0,processed posterior


,active_fti_run,posterior_source,number_of_parameters,parameters,number_of_samples_for_statistics,number_of_samples_plotted,plot_file,pdf_plot_file
0,fti_dchi0v,processed,12,"Mchirp,q,chiPN,chim,Deltat,dist,inc,phi,lambda...",40000,30000,/pbs/home/a/amahdi/PE_for_my_work/config_files...,/pbs/home/a/amahdi/PE_for_my_work/config_files...
1,fti_dchi1v,processed,12,"Mchirp,q,chiPN,chim,Deltat,dist,inc,phi,lambda...",35584,30000,/pbs/home/a/amahdi/PE_for_my_work/config_files...,/pbs/home/a/amahdi/PE_for_my_work/config_files...
2,fti_dchi2v,processed,12,"Mchirp,q,chiPN,chim,Deltat,dist,inc,phi,lambda...",40000,30000,/pbs/home/a/amahdi/PE_for_my_work/config_files...,/pbs/home/a/amahdi/PE_for_my_work/config_files...
3,fti_dchi3v,processed,12,"Mchirp,q,chiPN,chim,Deltat,dist,inc,phi,lambda...",35584,30000,/pbs/home/a/amahdi/PE_for_my_work/config_files...,/pbs/home/a/amahdi/PE_for_my_work/config_files...
4,fti_dchi4v,processed,12,"Mchirp,q,chiPN,chim,Deltat,dist,inc,phi,lambda...",40000,30000,/pbs/home/a/amahdi/PE_for_my_work/config_files...,/pbs/home/a/amahdi/PE_for_my_work/config_files...
5,fti_dchi6lv,processed,12,"Mchirp,q,chiPN,chim,Deltat,dist,inc,phi,lambda...",40000,30000,/pbs/home/a/amahdi/PE_for_my_work/config_files...,/pbs/home/a/amahdi/PE_for_my_work/config_files...
6,fti_dchi6v,processed,12,"Mchirp,q,chiPN,chim,Deltat,dist,inc,phi,lambda...",35584,30000,/pbs/home/a/amahdi/PE_for_my_work/config_files...,/pbs/home/a/amahdi/PE_for_my_work/config_files...
7,fti_dchi7v,processed,12,"Mchirp,q,chiPN,chim,Deltat,dist,inc,phi,lambda...",35584,30000,/pbs/home/a/amahdi/PE_for_my_work/config_files...,/pbs/home/a/amahdi/PE_for_my_work/config_files...
8,fti_dchiMinus2v,processed,12,"Mchirp,q,chiPN,chim,Deltat,dist,inc,phi,lambda...",35584,30000,/pbs/home/a/amahdi/PE_for_my_work/config_files...,/pbs/home/a/amahdi/PE_for_my_work/config_files...
9,fti_dkappa_S,processed,12,"Mchirp,q,chiPN,chim,Deltat,dist,inc,phi,lambda...",35584,30000,/pbs/home/a/amahdi/PE_for_my_work/config_files...,/pbs/home/a/amahdi/PE_for_my_work/config_files...


## Verifying the generated outputs

Listing only the files created under `normalized_bias_corner_plots` and confirming that the existing corner-plot directories remain untouched.


In [8]:
print('This notebook writes only under:')
print(NORMALIZED_CORNER_ROOT)
print()

created_files = sorted(
    path
    for path in NORMALIZED_CORNER_ROOT.rglob('*')
    if path.is_file()
)

print('Number of files in the new folder:', len(created_files))

for path in created_files:
    print(path)

if not created_files:
    print(
        '[warning] the output folder is empty. '
        'Review the warnings printed above.'
    )


This notebook writes only under:
/pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_6/normalized_bias_corner_plots

Number of files in the new folder: 25
/pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_6/normalized_bias_corner_plots/full_corner_plots/fti_dchi0v_all_parameters_normalized_bias_corner_processed.pdf
/pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_6/normalized_bias_corner_plots/full_corner_plots/fti_dchi0v_all_parameters_normalized_bias_corner_processed.png
/pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_6/normalized_bias_corner_plots/full_corner_plots/fti_dchi1v_all_parameters_normalized_bias_corner_processed.pdf
/pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_6/normalized_bias_corner_plots/full_corner_plots/fti_dchi1v_all_parameters_normalized_bias_corner_processed.png
/pbs/home/a/amahdi/PE_for_my_work/config_files/M1e6_SEOBv5_id_6/normalized_bias_corner_plots/full_corner_plots/fti_dchi2v_all_parameters_no